<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/hands_on_ml_with_scikit-learn_Aurelien_textbook/Chapter_7_Ensemble_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
%matplotlib

Using matplotlib backend: <object object at 0x7fc33d2606c0>


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [ ]:
from sklearn.datasets import make_moons

In [ ]:
dataset = make_moons(n_samples=10000, noise=0.4, random_state=42)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(dataset[0], dataset[1], test_size=0.1)

In [ ]:
log_clf = LogisticRegression()
rnd_clf = RandomForestClassifier()
svm_clf = SVC()

In [ ]:
voting_clf = VotingClassifier(
    estimators=[('lr', log_clf),
               ('rf', rnd_clf),
               ('svc', svm_clf)],
    voting='hard'
)

In [ ]:
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.825
RandomForestClassifier 0.834
SVC 0.85
VotingClassifier 0.846


In [ ]:
voting_clf = VotingClassifier(
        estimators=[
        ("lr", LogisticRegression()),
        ("rf", RandomForestClassifier()),
        ("svm", SVC(probability=True)) # probability
        #True makes svc estimate class probabilities (this
#        uses cross-validation)
        ],
        voting="soft" # when voting is soft, please ensure
#     all classifiers can estimate class probabilities.

)

In [ ]:
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.825
RandomForestClassifier 0.833
SVC 0.85
VotingClassifier 0.843


### Bagging and Pasting

In [ ]:
from matplotlib.colors import ListedColormap

In [ ]:
def plot_dataset(X, y):
    if X.shape[1] == 1:
        plt.plot(X, y, "bs")
    else:
        plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs")
        plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^")
    plt.grid(True, which='both')
    plt.xlabel(r"$x_1$", fontsize=20)
    plt.ylabel(r"$x_2$", fontsize=20, rotation=0)

def plot_predictions(train_X, train_y, clf, axes=[-1.5, 2.5, -1, 1.5]):
    plt.figure()
    plot_dataset(train_X, train_y)
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X).reshape(x0.shape)
    y_decision = clf.decision_function(X).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)
    plt.contourf(x0, x1, y_decision, cmap=plt.cm.brg, alpha=0.1)


def plot_decision_boundary(clf, X, y, axes=[-1.5, 2.5, -1, 1.5], alpha=0.5, contour=True):
    x1s = np.linspace(axes[0], axes[1], 100)
    x2s = np.linspace(axes[2], axes[3], 100)
    x1, x2 = np.meshgrid(x1s, x2s)
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)
    custom_cmap = ListedColormap(['#fafab0','#9898ff','#a0faa0'])
    plt.contourf(x1, x2, y_pred, alpha=0.3, cmap=custom_cmap)
    if contour:
        custom_cmap2 = ListedColormap(['#7d7d58','#4c4c7f','#507d50'])
        plt.contour(x1, x2, y_pred, cmap=custom_cmap2, alpha=0.8)
    plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", alpha=alpha)
    plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", alpha=alpha)
    plt.axis(axes)
    plt.xlabel(r"$x_1$", fontsize=18)
    plt.ylabel(r"$x_2$", fontsize=18, rotation=0)

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
bag_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=2),
                           n_estimators=500,
                           max_samples=100,
                           bootstrap=True, n_jobs=-1)
paste_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=2),
                           n_estimators=500,
                           max_samples=100,
                           bootstrap=False, n_jobs=-1)

In [ ]:
tree_clf = DecisionTreeClassifier(max_depth=2)

In [ ]:
tree_clf.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=9, max_leaf_nodes=20, min_samples_leaf=20)

In [ ]:
bag_clf.fit(X_train, y_train)

BaggingClassifier(base_estimator=DecisionTreeClassifier(max_depth=2),
                  max_samples=100, n_estimators=500, n_jobs=-1)

In [ ]:
paste_clf.fit(X_train, y_train)

BaggingClassifier(base_estimator=DecisionTreeClassifier(max_depth=2),
                  bootstrap=False, max_samples=100, n_estimators=500,
                  n_jobs=-1)

In [ ]:
plt.figure()
plot_decision_boundary(paste_clf, X_train, y_train)
plt.title("Decision Tree", fontsize=14)

Text(0.5, 1.0, 'Decision Tree')

In [ ]:
%matplotlib

Using matplotlib backend: TkAgg


In [ ]:
y_pred = bag_clf.predict(X_test)

### Out of Bag Evaluation

In [ ]:
bag_clf = BaggingClassifier(DecisionTreeClassifier(),
                           n_estimators=500,
                           bootstrap=True, n_jobs=-1,
                           oob_score=True)

In [ ]:
bag_clf.fit(X_train, y_train)

BaggingClassifier(base_estimator=DecisionTreeClassifier(), n_estimators=500,
                  n_jobs=-1, oob_score=True)

In [ ]:
bag_clf.oob_score_

0.8434444444444444

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
y_pred = bag_clf.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)

0.83

In [ ]:
bag_clf.oob_decision_function_

array([[1.        , 0.        ],
       [0.28795812, 0.71204188],
       [0.        , 1.        ],
       ...,
       [0.06878307, 0.93121693],
       [0.18032787, 0.81967213],
       [0.73298429, 0.26701571]])

### Random Patches and Random Subspaces

### Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rnd_clf = RandomForestClassifier(n_estimators=500,
                                max_leaf_nodes=16,
                                n_jobs=-1)

In [ ]:
rnd_clf.fit(X_train, y_train)

RandomForestClassifier(max_leaf_nodes=16, n_estimators=500, n_jobs=-1)

In [ ]:
y_pred_rf = rnd_clf.predict(X_test)

In [ ]:
plot_decision_boundary(rnd_clf, X_train, y_train)

### Extra Trees

### Feature Importance

In [ ]:
from sklearn.datasets import load_iris

In [ ]:
iris = load_iris()

In [ ]:
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1)

rnd_clf.fit(iris["data"], iris["target"])

In [ ]:
for name, score in zip(iris["feature_names"],
                      rnd_clf.feature_importances_):
    print(name, score*100, "%")

sepal length (cm) 9.266922267765318 %
sepal width (cm) 2.5873904472119267 %
petal length (cm) 42.21123752650283 %
petal width (cm) 45.93444975851992 %


### Boosting

### AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    algorithm="SAMME.R",
    learning_rate=0.5)

In [ ]:
ada_clf.fit(X_train, y_train)

AdaBoostClassifier(base_estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=200)

In [ ]:
plot_decision_boundary(ada_clf, X_train, y_train)

In [ ]:
y_pred_ada = ada_clf.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred_ada)

0.846

### Gradient Boosting

In [ ]:
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
tree_reg1 = DecisionTreeRegressor(max_depth=2)
tree_reg1.fit(X, y)

DecisionTreeRegressor(max_depth=2)

In [ ]:
# next we train a second DecisionTreeRegressor on the
# residual errors made by the first predictor
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2)
tree_reg2.fit(X, y2)

DecisionTreeRegressor(max_depth=2)

In [ ]:
# next we train a third DecisionTreeRegressor on the
# residual errors made by the second predictor
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2)
tree_reg3.fit(X, y3)

DecisionTreeRegressor(max_depth=2)

In [ ]:
# next we train a fourth DecisionTreeRegressor on the
# residual errors made by the third predictor
y4 = y3 - tree_reg3.predict(X)
tree_reg4 = DecisionTreeRegressor(max_depth=2)
tree_reg4.fit(X, y4)

DecisionTreeRegressor(max_depth=2)

In [ ]:
# Now we have an ensemble containing three trees.
# we can make predictions on a new instance simply by
# adding up the predictions of all the trees
y_pred_tree_regs = sum(tree.predict(X) for tree in (tree_reg1, tree_reg2, tree_reg3, tree_reg4))

In [ ]:
def plot_predictions(regressors, X, y, axes, label=None, style="r-", data_style="b.", data_label=None):
    x1 = np.linspace(axes[0], axes[1], 500)
    y_pred = sum(regressor.predict(x1.reshape(-1, 1)) for regressor in regressors)
    plt.plot(X[:, 0], y, data_style, label=data_label)
    plt.plot(x1, y_pred, style, linewidth=2, label=label)
    if label or data_label:
        plt.legend(loc="upper center", fontsize=16)
    plt.axis(axes)

plt.figure(figsize=(11,11))

plt.subplot(321)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h_1(x_1)$", style="g-", data_label="Training set")
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.title("Residuals and tree predictions", fontsize=16)

plt.subplot(322)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1)$", data_label="Training set")
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.title("Ensemble predictions", fontsize=16)

plt.subplot(323)
plot_predictions([tree_reg2], X, y2, axes=[-0.5, 0.5, -0.5, 0.5], label="$h_2(x_1)$", style="g-", data_style="k+", data_label="Residuals")
plt.ylabel("$y - h_1(x_1)$", fontsize=16)

plt.subplot(324)
plot_predictions([tree_reg1, tree_reg2], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1) + h_2(x_1)$")
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.subplot(325)
plot_predictions([tree_reg3], X, y3, axes=[-0.5, 0.5, -0.5, 0.5], label="$h_3(x_1)$", style="g-", data_style="k+")
plt.ylabel("$y - h_1(x_1) - h_2(x_1)$", fontsize=16)
plt.xlabel("$x_1$", fontsize=16)

plt.subplot(326)
plot_predictions([tree_reg1, tree_reg2, tree_reg3], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1) + h_2(x_1) + h_3(x_1)$")
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.figure()
plt.subplot(321)
plot_predictions([tree_reg4], X, y4, axes=[-0.5, 0.5, -0.5, 0.5], label="$h_3(x_1)$", style="g-", data_style="k+")
plt.ylabel("$y - h_1(x_1) - h_2(x_1) - h_3(x_1)$", fontsize=16)
plt.xlabel("$x_1$", fontsize=16)

plt.subplot(322)
plot_predictions([tree_reg1, tree_reg2, tree_reg3, tree_reg4], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1) + h_2(x_1) + h_3(x_1)$")
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.show()

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
gbrt = GradientBoostingRegressor(max_depth=2,
                                n_estimators=6,
                                learning_rate=1.0)

In [ ]:
gbrt.fit(X, y)

GradientBoostingRegressor(max_depth=2, n_estimators=6)

In [ ]:
plt.figure()
plot_predictions([gbrt], X, y, axes=[-0.5, 0.5, -0.1, 0.8])
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

Text(0, 0.5, '$y$')

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y)


In [ ]:
gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, learning_rate=0.1)
gbrt.fit(X_train, y_train)


GradientBoostingRegressor(max_depth=2, n_estimators=120)

In [ ]:
errors = [mean_squared_error(y_val, y_pred)
         for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators = np.argmin(errors) + 1

In [ ]:
bst_n_estimators

In [ ]:
gbrt_best = GradientBoostingRegressor(max_depth=2,
                n_estimators=bst_n_estimators, learning_rate=0.1)
gbrt_best.fit(X, y)

GradientBoostingRegressor(max_depth=2, n_estimators=79)

In [ ]:
plt.figure()
plot_predictions([gbrt_best], X, y, axes=[-0.5, 0.5, -0.1, 0.8])
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

Text(0, 0.5, '$y$')

In [ ]:
plt.plot(list(range(0, 120)), errors)

In [ ]:
gbrt = GradientBoostingRegressor(max_depth=2, warm_start=True)

min_val_error = float("inf")
error_going_up = 0

In [ ]:
for n_estimators in range(1, 120):
    gbrt.n_estimators = n_estimators
    gbrt.fit(X_train, y_train)
    y_pred = gbrt.predict(X_val)
    val_error = mean_squared_error(y_val, y_pred)
    if val_error < min_val_error:
        min_val_error = val_error
        error_going_up = 0
    else:
        error_going_up += 1
        if error_going_up == 15:
            break

In [ ]:
# optimal number of n_estimators
gbrt.n_estimators - error_going_up

79

### Stacking